# ForecastLLM - Week 8 Day 2: Gasoline Forecast Alerts


Donner pricing app: detect deals/opportunities  
ForecastLLM: detect meaningful gasoline price changes or forecast alerts


In [1]:
import json
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from week8.forecast_case import case_from_row, result_from_forecast
from week8.gasoline_loader import load_gasoline_series


In [2]:
load_dotenv()

gas_df = load_gasoline_series(series_id="GASREGW")

required_cols = {"timestamp", "value", "series_id", "source", "description", "unit"}
missing = required_cols - set(gas_df.columns)
if missing:
    raise ValueError(f"Gasoline dataframe missing required columns: {sorted(missing)}")

gas_df = gas_df.copy()
gas_df["timestamp"] = pd.to_datetime(gas_df["timestamp"], errors="coerce")
gas_df["value"] = pd.to_numeric(gas_df["value"], errors="coerce")
gas_df = gas_df.dropna(subset=["timestamp", "value"]).sort_values("timestamp").reset_index(drop=True)

if len(gas_df) < 60:
    raise RuntimeError(f"Need at least 60 weekly rows for alert workflow, got {len(gas_df)}")

gas_df.head()


,timestamp,value,series_id,source,description,unit
0,1990-08-20,1.191,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
1,1990-08-27,1.245,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
2,1990-09-03,1.242,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
3,1990-09-10,1.252,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon
4,1990-09-17,1.266,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon


In [3]:
supervised_df = gas_df[["timestamp", "value", "series_id", "source", "description", "unit"]].copy()
supervised_df["lag_1"] = supervised_df["value"].shift(1)
supervised_df["lag_2"] = supervised_df["value"].shift(2)
supervised_df["lag_4"] = supervised_df["value"].shift(4)
if len(supervised_df) > 52:
    supervised_df["lag_52"] = supervised_df["value"].shift(52)

iso_week = supervised_df["timestamp"].dt.isocalendar().week
supervised_df["week_of_year"] = iso_week.astype("int64")
supervised_df["month"] = supervised_df["timestamp"].dt.month.astype("int64")

supervised_df = supervised_df.dropna().reset_index(drop=True)
if supervised_df.empty:
    raise RuntimeError("No supervised rows available after lag construction.")

n = len(supervised_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)
train_df = supervised_df.iloc[:train_end].copy()
val_df = supervised_df.iloc[train_end:val_end].copy()
test_df = supervised_df.iloc[val_end:].copy()

if test_df.empty:
    raise RuntimeError("Test split is empty; cannot build forecast alerts.")

print(f"Rows={n:,} | train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")
supervised_df.head()


Rows=1,805 | train=1,263, val=271, test=271


,timestamp,value,series_id,source,description,unit,lag_1,lag_2,lag_4,lag_52,week_of_year,month
0,1991-09-30,1.092,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.097,1.110,1.127,1.191,40,9
1,1991-10-07,1.089,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.092,1.097,1.120,1.245,41,10
2,1991-10-14,1.084,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.089,1.092,1.110,1.242,42,10
3,1991-10-21,1.088,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.084,1.089,1.097,1.252,43,10
4,1991-10-28,1.091,GASREGW,FRED/EIA,U.S. Regular All Formulations Gasoline Price (...,dollars_per_gallon,1.088,1.084,1.092,1.266,44,10


In [4]:
test_eval = test_df.copy()
test_eval["naive_forecast"] = test_eval["lag_1"]
test_eval["seasonal_4_forecast"] = test_eval["lag_4"]

if "lag_52" in test_eval.columns:
    test_eval["seasonal_52_forecast"] = test_eval["lag_52"]
    test_eval["forecast"] = test_eval["seasonal_52_forecast"]
    test_eval["baseline_used"] = "lag_52"
else:
    test_eval["forecast"] = test_eval["seasonal_4_forecast"]
    test_eval["baseline_used"] = "lag_4"

if test_eval["forecast"].isna().any():
    test_eval["forecast"] = test_eval["naive_forecast"]
    test_eval.loc[test_eval["baseline_used"].isna(), "baseline_used"] = "lag_1"

test_eval["change"] = test_eval["forecast"] - test_eval["lag_1"]

def classify_alert(change: float, threshold: float = 0.05) -> str:
    if abs(change) < threshold:
        return "no_alert"
    return "increase_alert" if change > 0 else "decrease_alert"

test_eval["alert_type"] = test_eval["change"].map(classify_alert)

feature_keys = ["lag_1", "lag_2", "lag_4", "week_of_year", "month"]
if "lag_52" in supervised_df.columns:
    feature_keys.append("lag_52")

cases = []
results = []
for row in test_eval.itertuples(index=False):
    case = case_from_row(
        row,
        series_id=str(row.series_id),
        horizon=1,
        feature_keys=feature_keys,
        metadata={
            "split": "test",
            "source": str(row.source),
            "description": str(row.description),
            "unit": str(row.unit),
            "alert_threshold_dollars_per_gallon": 0.05,
            "alert_type": row.alert_type,
            "change": float(row.change),
            "baseline_used": str(row.baseline_used),
        },
    )
    result = result_from_forecast(
        case,
        forecast=float(row.forecast),
        model_name=f"baseline_{row.baseline_used}",
        notes=(
            f"alert_type={row.alert_type}; "
            f"change={float(row.change):.4f}; "
            f"threshold=0.05; baseline={row.baseline_used}"
        ),
    )
    cases.append(case)
    results.append(result)

inspection_df = test_eval[
    [
        "timestamp",
        "value",
        "lag_1",
        "forecast",
        "change",
        "alert_type",
        "baseline_used",
    ]
].rename(columns={"value": "actual", "lag_1": "last_observed"})
inspection_df.head(12)


,timestamp,actual,last_observed,forecast,change,alert_type,baseline_used
1534,2021-02-22,2.633,2.501,2.466,-0.035,no_alert,lag_52
1535,2021-03-01,2.711,2.633,2.423,-0.210,decrease_alert,lag_52
1536,2021-03-08,2.771,2.711,2.375,-0.336,decrease_alert,lag_52
1537,2021-03-15,2.853,2.771,2.248,-0.523,decrease_alert,lag_52
1538,2021-03-22,2.865,2.853,2.120,-0.733,decrease_alert,lag_52
1539,2021-03-29,2.852,2.865,2.005,-0.860,decrease_alert,lag_52
1540,2021-04-05,2.857,2.852,1.924,-0.928,decrease_alert,lag_52
1541,2021-04-12,2.849,2.857,1.853,-1.004,decrease_alert,lag_52
1542,2021-04-19,2.855,2.849,1.812,-1.037,decrease_alert,lag_52
1543,2021-04-26,2.872,2.855,1.773,-1.082,decrease_alert,lag_52


In [5]:
run_dir = Path("week8/run")
run_dir.mkdir(parents=True, exist_ok=True)

cases_path = run_dir / "week8_day2_cases.jsonl"
alerts_path = run_dir / "week8_day2_alerts.jsonl"

with cases_path.open("w", encoding="utf-8") as f:
    for case in cases:
        f.write(json.dumps(asdict(case), default=str) + "\n")

with alerts_path.open("w", encoding="utf-8") as f:
    for result in results:
        record = {
            "cutoff_timestamp": result.case.cutoff_timestamp,
            "series_id": result.case.series_id,
            "forecast": result.forecast,
            "actual": result.case.actual,
            "mae_vs_actual": result.mae_vs_actual,
            "smape_vs_actual": result.smape_vs_actual,
            "model_name": result.model_name,
            "alert_type": result.case.metadata.get("alert_type"),
            "change": result.case.metadata.get("change"),
            "alert_threshold_dollars_per_gallon": result.case.metadata.get(
                "alert_threshold_dollars_per_gallon"
            ),
            "baseline_used": result.case.metadata.get("baseline_used"),
            "notes": result.notes,
        }
        f.write(json.dumps(record, default=str) + "\n")

print(f"Wrote {len(cases):,} cases -> {cases_path}")
print(f"Wrote {len(results):,} alert rows -> {alerts_path}")

inspection_df.tail(10)


Wrote 271 cases -> week8/run/week8_day2_cases.jsonl
Wrote 271 alert rows -> week8/run/week8_day2_alerts.jsonl


,timestamp,actual,last_observed,forecast,change,alert_type,baseline_used
1795,2026-02-23,2.937,2.924,3.125,0.201,increase_alert,lag_52
1796,2026-03-02,3.015,2.937,3.078,0.141,increase_alert,lag_52
1797,2026-03-09,3.502,3.015,3.069,0.054,increase_alert,lag_52
1798,2026-03-16,3.720,3.502,3.058,-0.444,decrease_alert,lag_52
1799,2026-03-23,3.961,3.720,3.115,-0.605,decrease_alert,lag_52
1800,2026-03-30,3.990,3.961,3.162,-0.799,decrease_alert,lag_52
1801,2026-04-06,4.120,3.990,3.243,-0.747,decrease_alert,lag_52
1802,2026-04-13,4.123,4.120,3.168,-0.952,decrease_alert,lag_52
1803,2026-04-20,4.044,4.123,3.141,-0.982,decrease_alert,lag_52
1804,2026-04-27,4.123,4.044,3.133,-0.911,decrease_alert,lag_52


Alerting uses a fixed threshold (`abs(change) >= 0.05`) on purpose to keep behavior clear and debuggable while we establish the Week 8 workflow.

This notebook prepares the dataset, case objects, and alert artifacts for later agent/reporting steps.

No synthetic fallback is used. Missing local gasoline CSV data should fail loudly through `load_gasoline_series(...)`.
